# Capítulo 4 — Ablações de treinamento

Learning rate, scheduler, épocas, quantidade de pontos, batch size e, também, profundidade/largura da parte clássica da MPC híbrida. Clipping excluído.

**Fonte:** resultados do cluster, somente leitura. As bandas são **média ± desvio-padrão entre sementes** (não intervalos de confiança). Não misture o arquivo histórico da arquitetura antiga com o job novo de 5 qubits.

## 1. Configuração e funções comuns

Ajuste `SOURCE` na célula abaixo para `runs/jobs/ID/results` para a campanha nova. `archive` lê somente `results_*/` e ignora o diretório preliminar `results/`. As figuras e tabelas ficam em `runs/analysis/notebooks_tematicos/`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'runs').exists()), None)
if BASE is None:
    raise FileNotFoundError('Abra este notebook dentro do repositório que contém runs/.')
# Use 'archive' apenas para o estudo antigo. Use o job da NOVA arquitetura de 5 qubits:
# SOURCE = 'runs/jobs/2316/results'
SOURCE = 'archive'
ARCHIVE = BASE/'runs/archive/results'
SOURCE_PATH = ARCHIVE if SOURCE == 'archive' else BASE/SOURCE
FIGURES = BASE/'runs/analysis/notebooks_tematicos'
FIGURES.mkdir(parents=True, exist_ok=True)
PROBLEMS = {'burgers': ('Burgers', 'x', 't', 1.0),
            'black_scholes': ('Black–Scholes', 'S', 'τ', 40.0)}
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('Fonte:', SOURCE_PATH)

def get_records(problem, names):
    rows=[]
    for name in names:
        if SOURCE == 'archive':
            paths = sorted(SOURCE_PATH.glob(f'results_*/{problem}/{name}/seed_*/metrics.json'))
        else:
            paths = sorted((SOURCE_PATH/problem/name).glob('seed_*/metrics.json'))
        for p in paths:
            d=json.loads(p.read_text())
            rows.append({'problem':problem, 'model':name,
                         'seed':int(p.parent.name.replace('seed_', '')),
                         'folder':p.parent, **d})
    result=pd.DataFrame(rows)
    if not result.empty:
        result=result.drop_duplicates(['problem','model','seed'])
    return result

def metric_table(problem, names, save):
    d=get_records(problem,names)
    if d.empty:
        print('Ainda sem resultados:',problem,names)
        return d
    cols=[c for c in ['rmse','rmse_price','relative_l2','mae','time_s','parameters'] if c in d]
    groups=d.groupby('model')
    rows=[]
    for model, sub in groups:
        r={'model':model,'n':sub.seed.nunique()}
        for col in cols:
            if sub[col].notna().any():
                v=pd.to_numeric(sub[col],errors='coerce').dropna()
                r[col+'_mean']=v.mean();r[col+'_std']=v.std(ddof=1)
        rows.append(r)
    tab=pd.DataFrame(rows).sort_values('rmse_mean')
    display(tab.round(6))
    tab.to_csv(FIGURES/f'{save}_{problem}_metricas.csv',index=False)
    return d

def pair_table(problem, baseline, variants, save):
    d=get_records(problem,[baseline]+variants)
    if d.empty: return
    base=d[d.model==baseline][['seed','rmse']].rename(columns={'rmse':'base_rmse'})
    rows=[]
    for name in variants:
        test=d[d.model==name].merge(base,on='seed')
        if test.empty: continue
        delta=test.rmse-test.base_rmse
        rows.append({'variante':name,'n_pares':len(test),
                     'ΔRMSE_médio':delta.mean(),'ΔRMSE_DP':delta.std(ddof=1),
                     'melhores_seeds':int((delta<0).sum())})
    if rows:
        tab=pd.DataFrame(rows);display(tab.round(6));tab.to_csv(FIGURES/f'{save}_{problem}_pares.csv',index=False)

def load_grid(problem, name):
    rec=get_records(problem,[name]); frames=[]
    if rec.empty: return None
    for folder in rec.folder:
        f=folder/'predictions.csv'
        if f.exists():
            df=pd.read_csv(f).sort_values(['coordinate_1','coordinate_2']).reset_index(drop=True)
            frames.append(df)
    if not frames: return None
    ref=frames[0]
    x=np.sort(ref.coordinate_1.unique());t=np.sort(ref.coordinate_2.unique())
    if len(x)*len(t)!=len(ref):
        print('Malha de teste não retangular:',problem,name);return None
    coords=ref[['coordinate_1','coordinate_2']].to_numpy()
    valid=[f for f in frames if len(f)==len(ref) and
           np.allclose(f[['coordinate_1','coordinate_2']],coords,atol=1e-8,rtol=0)
           and np.allclose(f.exact,ref.exact,atol=1e-6,rtol=1e-5)]
    if len(valid)!=len(frames): print('Aviso: malhas diferentes; usadas',len(valid),'de',len(frames),'sementes')
    scale=PROBLEMS[problem][3]
    exact=(ref.exact.to_numpy()*scale).reshape(len(x),len(t))
    pred=np.stack([(f.prediction.to_numpy()*scale).reshape(len(x),len(t)) for f in valid])
    return x,t,exact,pred.mean(axis=0),pred.std(axis=0,ddof=1) if len(pred)>1 else np.zeros_like(exact),len(pred)

def surfaces(problem,name,save):
    d=load_grid(problem,name)
    if d is None:
        print('Sem predições:',problem,name);return
    x,t,exact,mean,sd,n=d
    panels=[exact,mean,np.abs(mean-exact),sd]
    labels=['Analítico','Predição média','Erro absoluto da média','DP entre sementes']
    fig,axes=plt.subplots(1,4,figsize=(17,4),constrained_layout=True)
    lo=min(exact.min(),mean.min());hi=max(exact.max(),mean.max())
    for ax,Z,label,i in zip(axes,panels,labels,range(4)):
        kw={'vmin':lo,'vmax':hi} if i<2 else {'vmin':0}
        im=ax.pcolormesh(x,t,Z.T,shading='auto',cmap='viridis' if i<2 else 'magma',**kw)
        ax.set(xlabel=PROBLEMS[problem][1],ylabel=PROBLEMS[problem][2],title=label)
        fig.colorbar(im,ax=ax)
    fig.suptitle(f'{PROBLEMS[problem][0]} · {name} · {n} seeds')
    fig.savefig(FIGURES/f'{save}_{problem}_superficies.png',dpi=160);plt.show()

def times(problem,name,save):
    d=load_grid(problem,name)
    if d is None:return
    x,t,exact,mean,sd,n=d
    fig,axes=plt.subplots(1,3,figsize=(15,3.8),sharey=True,constrained_layout=True)
    for ax,tm in zip(axes,[0,.5,1]):
        j=np.abs(t-tm).argmin()
        ax.plot(x,exact[:,j],color='black',lw=2,label='Analítico')
        ax.plot(x,mean[:,j],lw=1.6,label='Média da predição')
        ax.fill_between(x,mean[:,j]-sd[:,j],mean[:,j]+sd[:,j],alpha=.18,label='±1 DP')
        ax.set(xlabel=PROBLEMS[problem][1],title=f'{PROBLEMS[problem][2]} = {t[j]:.2f}')
        ax.grid(alpha=.2)
    axes[0].set_ylabel('Preço da opção' if problem=='black_scholes' else 'u(x,t)')
    axes[0].legend(fontsize=8)
    fig.suptitle(f'{PROBLEMS[problem][0]} · {name} · {n} seeds')
    fig.savefig(FIGURES/f'{save}_{problem}_tempos.png',dpi=160);plt.show()

def convergence(problem,names,save):
    fig,ax=plt.subplots(figsize=(9,4.5));found=False
    for name in names:
        rec=get_records(problem,[name]);hist=[]
        if rec.empty:continue
        for folder in rec.folder:
            f=folder/'history.csv'
            if f.exists():
                h=pd.read_csv(f)
                if {'epoch','val_mse'}.issubset(h.columns): hist.append(h.set_index('epoch')['val_mse'].rename(str(folder)))
        if not hist:continue
        v=pd.concat(hist,axis=1);mean=v.mean(axis=1);sd=v.std(axis=1,ddof=1).fillna(0)
        ax.plot(mean.index,mean.clip(lower=1e-12),label=f'{name} (n={len(hist)})')
        ax.fill_between(mean.index,(mean-sd).clip(lower=1e-12),(mean+sd).clip(lower=1e-12),alpha=.1)
        found=True
    if found:
        ax.set(yscale='log',xlabel='Época',ylabel='MSE de validação',title=f'{PROBLEMS[problem][0]} · média ± DP')
        ax.grid(alpha=.2);ax.legend(fontsize=8,ncol=2);fig.tight_layout()
        fig.savefig(FIGURES/f'{save}_{problem}_convergencia.png',dpi=160);plt.show()
    else:plt.close(fig)

def error_parameters(problem,names,save):
    d=get_records(problem,names)
    if d.empty:return
    g=d.groupby('model').agg(rmse=('rmse','mean'),sd=('rmse','std'),n=('seed','nunique'),params=('parameters','mean')).reset_index()
    fig,ax=plt.subplots(figsize=(8,4.5))
    for _,r in g.iterrows():
        ax.errorbar(r.params,r.rmse,yerr=r.sd if pd.notna(r.sd) else 0,fmt='o',capsize=4)
        ax.annotate(r.model,(r.params,r.rmse),xytext=(5,5),textcoords='offset points',fontsize=8)
    ax.set(xlabel='Parâmetros treináveis',ylabel='RMSE médio ± DP',yscale='log',title=f'{PROBLEMS[problem][0]} · erro × parâmetros')
    ax.grid(alpha=.2);fig.tight_layout();fig.savefig(FIGURES/f'{save}_{problem}_erro_parametros.png',dpi=160);plt.show()


## 2. Burgers

Métricas por modelo (média e DP), comparação pareada por semente e análise da arquitetura. Para Black–Scholes, as superfícies e os perfis são convertidos para **preço da opção**, multiplicando por `strike=40`; as métricas mantêm as unidades gravadas no arquivo.

### Learning rate e scheduler

Variantes analisadas: `mpc_hybrid`, `hybrid_lr_001`, `hybrid_lr_0001`, `hybrid_no_scheduler`, `hybrid_min_lr_0001`.

In [ ]:
names=['mpc_hybrid', 'hybrid_lr_001', 'hybrid_lr_0001', 'hybrid_no_scheduler', 'hybrid_min_lr_0001']
d = metric_table('burgers',names,'04_Treinamento')
pair_table('burgers','mpc_hybrid',['hybrid_lr_001', 'hybrid_lr_0001', 'hybrid_no_scheduler', 'hybrid_min_lr_0001'],'04_Treinamento')
error_parameters('burgers',names,'04_Treinamento')

### Épocas

Variantes analisadas: `mpc_hybrid`, `hybrid_2000epochs`, `hybrid_5000epochs`.

In [ ]:
names=['mpc_hybrid', 'hybrid_2000epochs', 'hybrid_5000epochs']
d = metric_table('burgers',names,'04_Treinamento')
pair_table('burgers','mpc_hybrid',['hybrid_2000epochs', 'hybrid_5000epochs'],'04_Treinamento')
error_parameters('burgers',names,'04_Treinamento')

### Dados e batch size

Variantes analisadas: `mpc_hybrid`, `hybrid_128points`, `hybrid_512points`, `hybrid_1000points`, `hybrid_batch_32`, `hybrid_batch_64`.

In [ ]:
names=['mpc_hybrid', 'hybrid_128points', 'hybrid_512points', 'hybrid_1000points', 'hybrid_batch_32', 'hybrid_batch_64']
d = metric_table('burgers',names,'04_Treinamento')
pair_table('burgers','mpc_hybrid',['hybrid_128points', 'hybrid_512points', 'hybrid_1000points', 'hybrid_batch_32', 'hybrid_batch_64'],'04_Treinamento')
error_parameters('burgers',names,'04_Treinamento')

### MLP principal da MPC híbrida

Variantes analisadas: `mpc_hybrid`, `hybrid_1layer`, `hybrid_2layers`, `hybrid_4layers`, `hybrid_16neurons`, `hybrid_32neurons`, `hybrid_64neurons`.

In [ ]:
names=['mpc_hybrid', 'hybrid_1layer', 'hybrid_2layers', 'hybrid_4layers', 'hybrid_16neurons', 'hybrid_32neurons', 'hybrid_64neurons']
d = metric_table('burgers',names,'04_Treinamento')
pair_table('burgers','mpc_hybrid',['hybrid_1layer', 'hybrid_2layers', 'hybrid_4layers', 'hybrid_16neurons', 'hybrid_32neurons', 'hybrid_64neurons'],'04_Treinamento')
error_parameters('burgers',names,'04_Treinamento')

### Superfícies, três tempos e convergência

As superfícies incluem a solução analítica, a média das predições, o erro absoluto da média e o DP entre sementes. Nos três tempos: `0`, `0.5` e `1`. Os gráficos de convergência são calculados sobre os históricos das sementes disponíveis.

In [ ]:
surfaces('burgers','mpc_hybrid','04_Treinamento')
times('burgers','mpc_hybrid','04_Treinamento')
convergence('burgers',['mpc_hybrid', 'hybrid_lr_001', 'hybrid_no_scheduler'],'04_Treinamento')

## 3. Black–Scholes

Métricas por modelo (média e DP), comparação pareada por semente e análise da arquitetura. Para Black–Scholes, as superfícies e os perfis são convertidos para **preço da opção**, multiplicando por `strike=40`; as métricas mantêm as unidades gravadas no arquivo.

### Learning rate e scheduler

Variantes analisadas: `mpc_hybrid`, `hybrid_lr_001`, `hybrid_lr_0001`, `hybrid_no_scheduler`, `hybrid_min_lr_0001`.

In [ ]:
names=['mpc_hybrid', 'hybrid_lr_001', 'hybrid_lr_0001', 'hybrid_no_scheduler', 'hybrid_min_lr_0001']
d = metric_table('black_scholes',names,'04_Treinamento')
pair_table('black_scholes','mpc_hybrid',['hybrid_lr_001', 'hybrid_lr_0001', 'hybrid_no_scheduler', 'hybrid_min_lr_0001'],'04_Treinamento')
error_parameters('black_scholes',names,'04_Treinamento')

### Épocas

Variantes analisadas: `mpc_hybrid`, `hybrid_2000epochs`, `hybrid_5000epochs`.

In [ ]:
names=['mpc_hybrid', 'hybrid_2000epochs', 'hybrid_5000epochs']
d = metric_table('black_scholes',names,'04_Treinamento')
pair_table('black_scholes','mpc_hybrid',['hybrid_2000epochs', 'hybrid_5000epochs'],'04_Treinamento')
error_parameters('black_scholes',names,'04_Treinamento')

### Dados e batch size

Variantes analisadas: `mpc_hybrid`, `hybrid_128points`, `hybrid_512points`, `hybrid_1000points`, `hybrid_batch_32`, `hybrid_batch_64`.

In [ ]:
names=['mpc_hybrid', 'hybrid_128points', 'hybrid_512points', 'hybrid_1000points', 'hybrid_batch_32', 'hybrid_batch_64']
d = metric_table('black_scholes',names,'04_Treinamento')
pair_table('black_scholes','mpc_hybrid',['hybrid_128points', 'hybrid_512points', 'hybrid_1000points', 'hybrid_batch_32', 'hybrid_batch_64'],'04_Treinamento')
error_parameters('black_scholes',names,'04_Treinamento')

### MLP principal da MPC híbrida

Variantes analisadas: `mpc_hybrid`, `hybrid_1layer`, `hybrid_2layers`, `hybrid_4layers`, `hybrid_16neurons`, `hybrid_32neurons`, `hybrid_64neurons`.

In [ ]:
names=['mpc_hybrid', 'hybrid_1layer', 'hybrid_2layers', 'hybrid_4layers', 'hybrid_16neurons', 'hybrid_32neurons', 'hybrid_64neurons']
d = metric_table('black_scholes',names,'04_Treinamento')
pair_table('black_scholes','mpc_hybrid',['hybrid_1layer', 'hybrid_2layers', 'hybrid_4layers', 'hybrid_16neurons', 'hybrid_32neurons', 'hybrid_64neurons'],'04_Treinamento')
error_parameters('black_scholes',names,'04_Treinamento')

### Superfícies, três tempos e convergência

As superfícies incluem a solução analítica, a média das predições, o erro absoluto da média e o DP entre sementes. Nos três tempos: `0`, `0.5` e `1`. Os gráficos de convergência são calculados sobre os históricos das sementes disponíveis.

In [ ]:
surfaces('black_scholes','mpc_hybrid','04_Treinamento')
times('black_scholes','mpc_hybrid','04_Treinamento')
convergence('black_scholes',['mpc_hybrid', 'hybrid_lr_001', 'hybrid_no_scheduler'],'04_Treinamento')

## 4. Interpretação

Compare **RMSE médio, variabilidade entre sementes, diferenças pareadas, número de parâmetros e custo computacional**. Uma diferença de médias não demonstra, sozinha, vantagem estatística ou quântica. Os painéis de previsão usam apenas sementes cujas malhas coincidem; a faixa ±DP descreve dispersão entre inicializações, não incerteza da solução analítica.